# Detección y clasificación de caracteres en documentos con OCR + CNN

- Objetivo: extraer texto de un documento con OCR clásico (Tesseract) y clasificar caracteres individuales con un modelo propio de deep learning.
- Caso de uso: digitalización de documentos oficiales (credenciales, formularios), donde el OCR genérico no siempre es suficiente.
- Los datos de documento usados aquí son especímenes públicos de muestra, no identidades reales. El dataset de caracteres por fuente es sintético, generado para este proyecto.
- Artículo completo, con el diagrama de la solución y el razonamiento detrás de cada decisión: [fuzzyfrog.ai/es/ai-lab/proyectos/gobierno/deteccion-clasificacion-caracteres-ocr-tesseract-cnn](https://fuzzyfrog.ai/es/ai-lab/proyectos/gobierno/deteccion-clasificacion-caracteres-ocr-tesseract-cnn/)


## Diagrama del pipeline

- **Entrada:** imagen del documento (escaneo o foto).
- **Etapa 1 — Detección (Tesseract):** localiza palabras y caracteres, produce cajas delimitadoras (bounding boxes).
- **Etapa 2 — Preprocesamiento:** escalado, suavizado y binarización antes de la detección, porque la resolución de la imagen afecta directamente la calidad del OCR.
- **Etapa 3 — Clasificación (CNN propia):** un clasificador entrenado con caracteres sintéticos por fuente (20×20 px) reconoce cada carácter recortado, como alternativa o refuerzo cuando el motor de OCR falla.
- **Salida:** texto estructurado del documento + confianza por carácter.


## Carga de datos

- Dos fuentes de datos en este proyecto: (1) imágenes de documento para la etapa de detección, (2) caracteres renderizados por fuente para entrenar el clasificador.
- Las imágenes de documento son especímenes de muestra públicos, sin datos personales reales.
- El dataset de caracteres por fuente es sintético: se genera renderizando cada letra (A-Z) y dígito (0-9) en dos tipografías, con variaciones de tamaño, posición y ruido — así el clasificador aprende variabilidad realista sin depender de un dataset externo.


In [ ]:
# Instalar dependencias (Colab)
!sudo apt-get install -y tesseract-ocr -q
!pip install pytesseract torch scikit-learn pandas pillow opencv-python-headless -q


In [ ]:
import pytesseract
import cv2
import re
import glob
import string
import random
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageDraw, ImageFont
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

import torch
import torch.nn as nn

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)


In [ ]:
# Cargar imagenes del documento (especimenes de muestra publicos)
imgs = []
nombres = []
for img in sorted(glob.glob("text_detector/*.png")):
    n = cv2.imread(img, 0)
    imgs.append(n)
    nombres.append(img)

print(f"Imagenes cargadas: {len(imgs)}")
for n, im in zip(nombres, imgs):
    print(n, im.shape)


In [ ]:
def mostrar_imgs(imgs, titulos=None):
    fig, axes = plt.subplots(1, len(imgs), figsize=(6*len(imgs), 6))
    if len(imgs) == 1:
        axes = [axes]
    for i, (img, ax) in enumerate(zip(imgs, axes)):
        ax.imshow(img, cmap="gray")
        if titulos:
            ax.set_title(titulos[i], fontsize=10)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

mostrar_imgs(imgs, nombres)


## Explicación de datos

- Cada imagen de documento tiene texto en distintas zonas (encabezado institucional, nombre, domicilio, folio) con distinta calidad de impresión y contraste.
- Antes de correr el OCR sobre la imagen cruda, conviene inspeccionar resolución y contraste — de eso depende directamente cuántas palabras se detectan correctamente.
- El desbalance típico en estos documentos: campos numéricos (folios, fechas) suelen tener mejor contraste que campos manuscritos o zonas con sello de fondo.


In [ ]:
# Inspeccionar resolucion y contraste de cada imagen
for n, im in zip(nombres, imgs):
    print(f"{n}: shape={im.shape}, contraste (std)={im.std():.2f}, brillo medio={im.mean():.2f}")


## Análisis de datos — primer intento de detección (sin preprocesar)

- Primer intento: correr Tesseract directo sobre la imagen cruda, sin ningún preprocesamiento.
- El resultado real fue más ruidoso de lo esperado — tokens sin sentido mezclados con palabras correctas. Esto se documenta tal cual salió, no se filtra para verse mejor.


In [ ]:
def detectar_palabras(imagen, config=r'--oem 1'):
    h_img, w_img = imagen.shape
    boxes = pytesseract.image_to_data(imagen, config=config)
    tokens = []
    for i, b in enumerate(boxes.splitlines()):
        if i == 0:
            continue
        b = b.split()
        if len(b) == 12 and re.match('[a-zA-Z0-9]', b[11]):
            tokens.append(b[11])
    return tokens

print("== Deteccion SIN preprocesar ==")
for n, im in zip(nombres, imgs):
    tokens = detectar_palabras(im)
    print(f"{n}: {len(tokens)} tokens detectados -> {tokens[:12]}")


**Resultado real:** en la muestra de menor contraste se detectaron tokens como `wwenico`, `CCREDENCIAL` — ruido de OCR sobre una imagen de baja resolución, no palabras reales. Esto confirma que el preprocesamiento no es opcional para este tipo de documento escaneado o fotografiado.

### Segundo intento: con preprocesamiento (upscaling + binarización)


In [ ]:
def preprocesar(imagen):
    im2 = cv2.resize(imagen, None, fx=2.5, fy=2.5, interpolation=cv2.INTER_CUBIC)
    im2 = cv2.GaussianBlur(im2, (3, 3), 0)
    _, im2 = cv2.threshold(im2, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return im2

print("== Deteccion CON preprocesamiento ==")
resultados_deteccion = {}
for n, im in zip(nombres, imgs):
    proc = preprocesar(im)
    tokens = detectar_palabras(proc)
    resultados_deteccion[n] = tokens
    print(f"{n}: {len(tokens)} tokens detectados -> {tokens[:12]}")


**Resultado real:** el preprocesamiento aumentó la cantidad de tokens detectados en ambas imágenes, pero también aumentó el ruido — más palabras correctas, pero también más fragmentos sin sentido. El preprocesamiento ayuda, no resuelve el problema por sí solo; ver la sección de iteración en el artículo para el detalle de esta decisión.


## Modelado — generación del dataset sintético y entrenamiento del clasificador

- Se genera un dataset sintético de caracteres (A-Z, 0-9) en dos tipografías, 20×20 px, con variación de tamaño, posición y ruido gaussiano — sin usar ningún dataset propietario.
- Se entrena primero un baseline simple (regresión logística sobre píxeles crudos) y después una CNN pequeña, para poder comparar con criterio, no solo entrenar el modelo más complejo por default.


In [ ]:
def render_char(ch, font_path, size=20, jitter=True):
    img = Image.new("L", (size, size), color=255)
    draw = ImageDraw.Draw(img)
    fsize = random.randint(14, 17) if jitter else 16
    font = ImageFont.truetype(font_path, fsize)
    bbox = draw.textbbox((0, 0), ch, font=font)
    w, h = bbox[2] - bbox[0], bbox[3] - bbox[1]
    dx = random.randint(-2, 2) if jitter else 0
    dy = random.randint(-2, 2) if jitter else 0
    x = (size - w) // 2 - bbox[0] + dx
    y = (size - h) // 2 - bbox[1] + dy
    draw.text((x, y), ch, font=font, fill=0)
    arr = np.array(img)
    if jitter:
        noise = np.random.normal(0, 6, arr.shape)
        arr = np.clip(arr.astype(float) + noise, 0, 255).astype(np.uint8)
    return arr

chars = list(string.ascii_uppercase) + list(string.digits)  # 36 clases
fonts = {
    "ARIAL":   "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",
    "CALIBRI": "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
}
N_PER_CLASS = 60

rows = []
for font_name, font_path in fonts.items():
    for ch in chars:
        for i in range(N_PER_CLASS):
            arr = render_char(ch, font_path, jitter=(i > 0))
            rows.append([ch] + arr.flatten().tolist())

cols = ["label"] + [f"p{i}" for i in range(400)]
data = pd.DataFrame(rows, columns=cols)
print(f"Dataset sintetico: {data.shape[0]} muestras, {len(chars)} clases")
data.to_csv("dataset/caracteres_fuentes_sintetico.csv", index=False)


In [ ]:
# Vista previa de algunas muestras del dataset sintetico
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
sample_idx = np.random.choice(len(data), 16, replace=False)
for ax, idx in zip(axes.flatten(), sample_idx):
    row = data.iloc[idx]
    img = row.drop("label").values.reshape(20, 20).astype(np.uint8)
    ax.imshow(img, cmap="gray")
    ax.set_title(row["label"], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
le = LabelEncoder()
y = le.fit_transform(data["label"].astype(str))
X = data.drop(columns=["label"]).values.astype(np.float32) / 255.0

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")


In [ ]:
# Baseline: regresion logistica sobre pixeles crudos
baseline = LogisticRegression(max_iter=1000)
baseline.fit(X_train, y_train)
baseline_acc = accuracy_score(y_test, baseline.predict(X_test))
print(f"Baseline (Regresion Logistica) accuracy: {baseline_acc:.4f}")


In [ ]:
# CNN pequena en PyTorch
class CharCNN(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.fc = nn.Sequential(
            nn.Flatten(), nn.Linear(32 * 5 * 5, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, n_classes)
        )

    def forward(self, x):
        return self.fc(self.net(x))

Xtr = torch.tensor(X_train).view(-1, 1, 20, 20)
Xte = torch.tensor(X_test).view(-1, 1, 20, 20)
ytr = torch.tensor(y_train, dtype=torch.long)
yte = torch.tensor(y_test, dtype=torch.long)

model = CharCNN(n_classes=len(le.classes_))
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
lossfn = nn.CrossEntropyLoss()

ds = torch.utils.data.TensorDataset(Xtr, ytr)
dl = torch.utils.data.DataLoader(ds, batch_size=64, shuffle=True)

model.train()
for epoch in range(15):
    tot_loss = 0
    for xb, yb in dl:
        opt.zero_grad()
        loss = lossfn(model(xb), yb)
        loss.backward()
        opt.step()
        tot_loss += loss.item()
    if epoch % 3 == 0 or epoch == 14:
        print(f"epoch {epoch:2d} | loss {tot_loss/len(dl):.4f}")


## Evaluación

- Se compara el baseline contra la CNN con la misma partición de test, nunca vista durante el entrenamiento.
- La pregunta que importa no es solo "qué modelo gana", sino cuánto justifica la complejidad extra de la CNN frente al baseline simple.


In [ ]:
model.eval()
with torch.no_grad():
    preds = model(Xte).argmax(dim=1)
cnn_acc = accuracy_score(yte.numpy(), preds.numpy())
print(f"CNN accuracy: {cnn_acc:.4f}")

comparacion = pd.DataFrame({
    "modelo": ["Regresion Logistica (baseline)", "CNN (2 conv + FC)"],
    "accuracy_test": [round(baseline_acc, 4), round(cnn_acc, 4)]
})
print(comparacion)
comparacion.to_csv("dataset/comparacion_modelos.csv", index=False)


In [ ]:
# Guardar modelo entrenado y encoder de etiquetas
torch.save(model.state_dict(), "dataset/char_cnn.pt")
with open("dataset/label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)
print("Modelo y encoder guardados.")


## Hallazgos principales

- El salto de 79.05% (regresión logística) a 99.19% (CNN) en el mismo dataset sintético confirma que la estructura espacial del carácter sí aporta señal que un modelo lineal sobre píxeles crudos no puede capturar — la complejidad extra de la CNN se justificó con datos, no por default.
- El OCR sobre la imagen cruda del documento produjo detecciones con ruido real (tokens sin sentido mezclados con palabras válidas): la calidad de imagen de entrada es, en la práctica, más determinante que el motor de OCR elegido.
- El preprocesamiento (upscaling + binarización) aumentó las detecciones totales pero no las "limpió" automáticamente — subió tanto las palabras correctas como el ruido. Preprocesar ayuda, pero no reemplaza una etapa de validación posterior.
- El dataset de caracteres sintético, generado por fuente sin depender de ningún dataset propietario, fue suficiente para entrenar un clasificador con exactitud alta — útil cuando no hay acceso a un dataset real etiquetado.
